# 02d Global Mamba Strict Forecasting

Train official `mamba-ssm` pooled/global Mamba sequence models with calendar-global purging and profit/risk validation selection.

## Setup

In [1]:
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
for candidate in [cwd, cwd / "forecasting", cwd.parent, cwd.parent / "forecasting"]:
    if (candidate / "src" / "stock_forecast").exists():
        PROJECT_DIR = candidate
        break
else:
    raise RuntimeError("Cannot locate forecasting project directory with src/stock_forecast")

SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

ARTIFACT_DIR = PROJECT_DIR / "artifacts"
DATA_DIR = ARTIFACT_DIR / "data"
REPORTS_DIR = ARTIFACT_DIR / "reports"
MAMBA_DATA_DIR = DATA_DIR / "mamba"
MAMBA_OUTPUT_DIR = Path(os.environ.get("MAMBA_OUTPUT_DIR", ARTIFACT_DIR / "horizons")).expanduser().resolve()
for path in [DATA_DIR, REPORTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("TILELANG_CACHE_DIR", "/tmp/tilelang")


def display_path(path: Path) -> str:
    try:
        return str(path.relative_to(PROJECT_DIR))
    except ValueError:
        return str(path)


print(f"PROJECT_DIR = {PROJECT_DIR}")

PROJECT_DIR = /home/sapce/forecasting_stock_prices/forecasting


In [2]:
import importlib.util
import pandas as pd
from IPython.display import display

from stock_forecast.artifacts import load_json, load_table
from stock_forecast.mlflow_tracking import MLflowRunConfig, log_strict_protocol_result, mlflow_horizon_run
from stock_forecast.models import build_model
from stock_forecast.strict_protocol import run_strict_global_lstm_protocol

pd.set_option("display.max_columns", 220)

if importlib.util.find_spec("torch") is None:
    raise ImportError("PyTorch is required for this notebook. Install/use the torchlab environment.")
if importlib.util.find_spec("mamba_ssm") is None:
    raise ImportError("Official mamba-ssm is required. Install PyTorch first, then `pip install \"mamba-ssm[causal-conv1d]\" --no-build-isolation`.")

import torch


def env_int(name: str, default: int) -> int:
    return int(os.environ.get(name, default))


def env_bool(name: str, default: bool) -> bool:
    value = os.environ.get(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "y"}


def env_int_list(name: str, default: list[int]) -> list[int]:
    value = os.environ.get(name)
    if not value:
        return default
    return [int(item.strip()) for item in value.split(",") if item.strip()]

## Constants

In [3]:
FORCE_RETRAIN = env_bool("MAMBA_FORCE_RETRAIN", True)
RANDOM_STATE = env_int("MAMBA_RANDOM_STATE", 42)
MAMBA_REQUIRE_CUDA = env_bool("MAMBA_REQUIRE_CUDA", True)

STRICT_VALIDATION_ROWS = env_int("STRICT_VALIDATION_ROWS", 126)
STRICT_TEST_ROWS = env_int("STRICT_TEST_ROWS", 126)
MATURE_MIN_ROWS = env_int("MATURE_MIN_ROWS", 1008)
LIMITED_HISTORY_MIN_BLOCK_ROWS = env_int("LIMITED_HISTORY_MIN_BLOCK_ROWS", 42)
MIN_TRAIN_ROWS = env_int("MIN_TRAIN_ROWS", 60)
STRICT_MAX_TRAIN_ROWS = env_int("STRICT_MAX_TRAIN_ROWS", 1260)
INNER_MAX_FOLDS = env_int("INNER_MAX_FOLDS", 3)
INNER_MIN_TRAIN_ROWS = env_int("INNER_MIN_TRAIN_ROWS", 504)

MAMBA_N_TRIALS = env_int("MAMBA_N_TRIALS", 30)
MAMBA_OPTUNA_N_JOBS = env_int("MAMBA_OPTUNA_N_JOBS", 1)
MAMBA_MAX_EPOCHS = env_int("MAMBA_MAX_EPOCHS", 120)
MAMBA_PATIENCE = env_int("MAMBA_PATIENCE", 12)
MAMBA_DEVICE = os.environ.get("MAMBA_DEVICE", "auto")
MAMBA_ENSEMBLE_SEEDS = env_int_list("MAMBA_ENSEMBLE_SEEDS", [1, 7, 21])

TRANSACTION_COST_BPS = 10
SLIPPAGE_BPS = 5
SIGNAL_ANCHOR = "expanding_median"
MIN_VALIDATION_TRADES = 8
MAX_VALIDATION_DRAWDOWN = -0.35

MLFLOW_ENABLED = env_bool("MLFLOW_ENABLED", True)
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000")
MLFLOW_EXPERIMENT_NAME = os.environ.get("MLFLOW_EXPERIMENT_NAME", "stock_return_forecasting_research")
MLFLOW_LOG_OPTUNA_TRIALS = env_bool("MLFLOW_LOG_OPTUNA_TRIALS", True)

HORIZONS = [
    {"name": "week", "horizon": 5, "threshold_grid": [0.0, 0.0025, 0.005, 0.01]},
    {"name": "month", "horizon": 21, "threshold_grid": [0.0, 0.005, 0.01, 0.02]},
]
selected_horizons = {name.strip() for name in os.environ.get("MAMBA_HORIZONS", "").split(",") if name.strip()}
if selected_horizons:
    HORIZONS = [item for item in HORIZONS if item["name"] in selected_horizons]

if MAMBA_REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError("Official mamba-ssm is CUDA-oriented. Run on a CUDA host or set MAMBA_REQUIRE_CUDA=0 for an explicit CPU experiment.")

## Model Config

In [4]:
def make_mamba_search_space(horizon: int) -> dict:
    lookbacks = [60, 90, 126, 252] if horizon >= 21 else [60, 90, 126]
    huber_beta_choices = [0.04, 0.08, 0.12] if horizon >= 21 else [0.02, 0.04, 0.06]
    return {
        "lookback": {"type": "categorical", "choices": lookbacks},
        "model_dim": {"type": "categorical", "choices": [32, 64, 96, 128]},
        "num_layers": {"type": "categorical", "choices": [1, 2, 3]},
        "d_state": {"type": "categorical", "choices": [8, 16, 32, 64]},
        "d_conv": {"type": "categorical", "choices": [2, 3, 4]},
        "expand": {"type": "categorical", "choices": [1, 2]},
        "block_dropout": {"type": "float", "low": 0.0, "high": 0.35},
        "head_dropout": {"type": "float", "low": 0.10, "high": 0.50},
        "learning_rate": {"type": "float", "low": 5e-5, "high": 2e-3, "log": True},
        "weight_decay": {"type": "float", "low": 1e-5, "high": 1e-2, "log": True},
        "batch_size": {"type": "categorical", "choices": [32, 64, 128]},
        "loss": {"type": "categorical", "choices": ["smooth_l1", "huber", "mse"]},
        "huber_beta": {"type": "categorical", "choices": huber_beta_choices},
        "feature_clip": {"type": "categorical", "choices": [3.0, 5.0, 8.0]},
        "target_normalization": {"type": "categorical", "choices": ["global", "per_ticker"]},
        "balanced_ticker_sampling": {"type": "categorical", "choices": [True]},
        "grad_clip_norm": {"type": "float", "low": 0.5, "high": 2.0},
    }


def make_mamba_config(name: str, feature_cols: list[str], horizon: int) -> dict:
    return {
        "name": name,
        "model_type": "mamba",
        "estimator_factory": build_model,
        "input_mode": "full_frame",
        "feature_cols": feature_cols,
        "static_params": {
            "max_epochs": MAMBA_MAX_EPOCHS,
            "patience": MAMBA_PATIENCE,
            "device": MAMBA_DEVICE,
            "validation_fraction": 0.2,
        },
        "search_space": make_mamba_search_space(horizon),
        "post_selection_static_params": {"ensemble_seeds": MAMBA_ENSEMBLE_SEEDS},
        "n_trials": MAMBA_N_TRIALS,
        "optuna_n_jobs": MAMBA_OPTUNA_N_JOBS,
        "needs_scaler": False,
    }

## Load Mamba Data

In [5]:
horizon_inputs = []

for spec in HORIZONS:
    horizon_name = spec["name"]
    horizon = int(spec["horizon"])
    mamba_dir = MAMBA_DATA_DIR / "horizons" / horizon_name
    model_path = mamba_dir / "model_dataset.parquet"
    feature_path = mamba_dir / "feature_columns.json"
    if not model_path.exists() and not model_path.with_suffix(".csv").exists():
        raise FileNotFoundError(f"Missing Mamba data for {horizon_name}. Run notebooks/01d_mamba_eda.ipynb first.")
    payload = load_json(feature_path)
    model_df = load_table(model_path)
    model_df["date"] = pd.to_datetime(model_df["date"])
    feature_sets = payload["mamba_feature_sets"]
    primary_feature_set = payload.get("primary_mamba_feature_set", "mamba_stationary")
    model_configs = [
        make_mamba_config("global_mamba_stationary", feature_sets["mamba_stationary"], horizon),
        make_mamba_config("global_mamba_all", feature_sets["mamba_all"], horizon),
    ]
    horizon_inputs.append({
        "horizon_name": horizon_name,
        "horizon": horizon,
        "threshold_grid": spec["threshold_grid"],
        "model_df": model_df,
        "feature_cols": feature_sets[primary_feature_set],
        "target_col": payload["target_column"],
        "model_configs": model_configs,
        "artifact_dir": MAMBA_OUTPUT_DIR / horizon_name / "global_mamba",
    })
    print({
        "horizon": horizon_name,
        "rows": len(model_df),
        "tickers": model_df["ticker"].nunique(),
        "mamba_all_features": len(feature_sets["mamba_all"]),
        "mamba_stationary_features": len(feature_sets["mamba_stationary"]),
        "target": payload["target_column"],
        "artifact_dir": display_path(MAMBA_OUTPUT_DIR / horizon_name / "global_mamba"),
    })

{'horizon': 'week', 'rows': 19407, 'tickers': 7, 'mamba_all_features': 275, 'mamba_stationary_features': 258, 'target': 'target_return_5_next_open', 'artifact_dir': 'artifacts/horizons/week/global_mamba'}
{'horizon': 'month', 'rows': 19295, 'tickers': 7, 'mamba_all_features': 275, 'mamba_stationary_features': 258, 'target': 'target_return_21_next_open', 'artifact_dir': 'artifacts/horizons/month/global_mamba'}


## Train And Evaluate

In [6]:
global_mamba_results = {}

for item in horizon_inputs:
    horizon_name = item["horizon_name"]
    horizon = int(item["horizon"])
    mlflow_config = MLflowRunConfig(
        tracking_uri=MLFLOW_TRACKING_URI,
        experiment_name=MLFLOW_EXPERIMENT_NAME,
        notebook_name="02d_mamba_forecasting",
        horizon_name=horizon_name,
        horizon=horizon,
        enabled=MLFLOW_ENABLED,
        log_optuna_trials=MLFLOW_LOG_OPTUNA_TRIALS,
    )
    mlflow_params = {
        "force_retrain": FORCE_RETRAIN,
        "random_state": RANDOM_STATE,
        "mamba_n_trials": MAMBA_N_TRIALS,
        "mamba_optuna_n_jobs": MAMBA_OPTUNA_N_JOBS,
        "mamba_max_epochs": MAMBA_MAX_EPOCHS,
        "mamba_patience": MAMBA_PATIENCE,
        "mamba_ensemble_seeds": MAMBA_ENSEMBLE_SEEDS,
        "model_count": len(item["model_configs"]),
        "feature_count": len(item["feature_cols"]),
        "target_col": item["target_col"],
        "threshold_grid": item["threshold_grid"],
        "validation_rows": STRICT_VALIDATION_ROWS,
        "test_rows": STRICT_TEST_ROWS,
        "transaction_cost_bps": TRANSACTION_COST_BPS,
        "slippage_bps": SLIPPAGE_BPS,
        "signal_anchor": SIGNAL_ANCHOR,
        "min_validation_trades": MIN_VALIDATION_TRADES,
        "max_validation_drawdown": MAX_VALIDATION_DRAWDOWN,
    }
    with mlflow_horizon_run(
        mlflow_config,
        params=mlflow_params,
        tags={"training_protocol": "strict_global_mamba", "training_notebook": "02d_mamba_forecasting"},
    ) as mlflow_run:
        result = run_strict_global_lstm_protocol(
            model_df=item["model_df"],
            feature_cols=item["feature_cols"],
            target_col=item["target_col"],
            model_configs=item["model_configs"],
            artifact_dir=item["artifact_dir"],
            force_retrain=FORCE_RETRAIN,
            random_state=RANDOM_STATE,
            run_metadata={
                "horizon_name": horizon_name,
                "horizon": horizon,
                "training_notebook": "02d_mamba_forecasting",
                "global_mamba_run": True,
            },
            validation_rows=STRICT_VALIDATION_ROWS,
            test_rows=STRICT_TEST_ROWS,
            mature_min_rows=MATURE_MIN_ROWS,
            limited_history_min_block_rows=LIMITED_HISTORY_MIN_BLOCK_ROWS,
            min_train_rows=MIN_TRAIN_ROWS,
            max_train_rows=STRICT_MAX_TRAIN_ROWS,
            inner_max_folds=INNER_MAX_FOLDS,
            inner_min_train_rows=INNER_MIN_TRAIN_ROWS,
            transaction_cost_bps=TRANSACTION_COST_BPS,
            slippage_bps=SLIPPAGE_BPS,
            threshold_grid=item["threshold_grid"],
            signal_anchor=SIGNAL_ANCHOR,
            min_validation_trades=MIN_VALIDATION_TRADES,
            max_validation_drawdown=MAX_VALIDATION_DRAWDOWN,
            mlflow_trial_logger=mlflow_run.trial_logger,
        )
        log_strict_protocol_result(
            result,
            params=mlflow_params,
            tags={"training_protocol": "strict_global_mamba", "training_notebook": "02d_mamba_forecasting"},
        )
    global_mamba_results[horizon_name] = result

    print(f"=== Global Mamba strict protocol: {horizon_name} ({horizon} trading days) ===")
    display(result["validation_threshold_search"])
    display(result["validation_model_ranking"])
    display(result["test_panel_signal_metrics"])
    display(result["test_signal_metrics"])
    display(result["leakage_audit"])

    failed = result["leakage_audit"][~result["leakage_audit"]["passed"]]
    if not failed.empty:
        raise AssertionError(f"Global Mamba leakage audit failed for {horizon_name}: {failed[check].tolist()}")

🏃 View run week-global_mamba_stationary-trial-0 at: http://localhost:5000/#/experiments/2/runs/5e8848e7d0134e558552eb0662516692
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run week-global_mamba_stationary-trial-1 at: http://localhost:5000/#/experiments/2/runs/eb5ec45d57ee47fe97d2e062010f5108
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run week-global_mamba_stationary-trial-2 at: http://localhost:5000/#/experiments/2/runs/096fab2276ae4ca69a0a2e494da92b5f
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run week-global_mamba_stationary-trial-3 at: http://localhost:5000/#/experiments/2/runs/bfc7670052d243539e43fa1ddad8a8cf
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run week-global_mamba_stationary-trial-4 at: http://localhost:5000/#/experiments/2/runs/15c1ee9b68054009962934a9b4aab6e0
🧪 View experiment at: http://localhost:5000/#/experiments/2
🏃 View run week-global_mamba_stationary-trial-5 at: http://l

,profit_risk_utility,panel_sharpe,mean_ticker_sharpe,cumulative_return,max_drawdown,turnover,number_of_trades,validation_constraints_pass,long_threshold,model_name,selected_threshold,horizon_name,horizon,training_notebook,global_mamba_run
0,-0.624211,-0.502172,-0.754022,-0.009205,-0.031037,0.023436,95,True,0.0100,global_mamba_stationary,0.01,week,5,02d_mamba_forecasting,True
1,-0.747214,-0.613666,-0.863693,-0.015415,-0.054303,0.035861,167,True,0.0050,global_mamba_stationary,0.01,week,5,02d_mamba_forecasting,True
2,-0.864000,-0.899596,-0.874004,-0.024434,-0.060454,0.043523,185,True,0.0025,global_mamba_stationary,0.01,week,5,02d_mamba_forecasting,True
3,-1.082900,-1.322229,-0.958530,-0.043233,-0.078559,0.049567,215,True,0.0000,global_mamba_stationary,0.01,week,5,02d_mamba_forecasting,True
4,-1.037849,-1.037417,-1.158043,-0.011687,-0.034721,0.023778,110,True,0.0100,global_mamba_all,0.01,week,5,02d_mamba_forecasting,True
5,-1.132054,-1.673607,-0.882045,-0.031250,-0.045586,0.045409,185,True,0.0050,global_mamba_all,0.01,week,5,02d_mamba_forecasting,True
6,-1.244965,-2.173627,-0.711055,-0.058893,-0.067268,0.052873,227,True,0.0000,global_mamba_all,0.01,week,5,02d_mamba_forecasting,True
7,-1.575092,-2.534032,-1.099988,-0.057809,-0.057532,0.044538,191,True,0.0025,global_mamba_all,0.01,week,5,02d_mamba_forecasting,True


,profit_risk_utility,panel_sharpe,mean_ticker_sharpe,cumulative_return,max_drawdown,turnover,number_of_trades,validation_constraints_pass,long_threshold,model_name,selected_threshold,horizon_name,horizon,training_notebook,global_mamba_run,validation_rank,is_validation_selected
0,-0.624211,-0.502172,-0.754022,-0.009205,-0.031037,0.023436,95,True,0.0100,global_mamba_stationary,0.01,week,5,02d_mamba_forecasting,True,1,True
1,-0.747214,-0.613666,-0.863693,-0.015415,-0.054303,0.035861,167,True,0.0050,global_mamba_stationary,0.01,week,5,02d_mamba_forecasting,True,2,False
2,-0.864000,-0.899596,-0.874004,-0.024434,-0.060454,0.043523,185,True,0.0025,global_mamba_stationary,0.01,week,5,02d_mamba_forecasting,True,3,False
4,-1.037849,-1.037417,-1.158043,-0.011687,-0.034721,0.023778,110,True,0.0100,global_mamba_all,0.01,week,5,02d_mamba_forecasting,True,4,False
3,-1.082900,-1.322229,-0.958530,-0.043233,-0.078559,0.049567,215,True,0.0000,global_mamba_stationary,0.01,week,5,02d_mamba_forecasting,True,5,False
5,-1.132054,-1.673607,-0.882045,-0.031250,-0.045586,0.045409,185,True,0.0050,global_mamba_all,0.01,week,5,02d_mamba_forecasting,True,6,False
6,-1.244965,-2.173627,-0.711055,-0.058893,-0.067268,0.052873,227,True,0.0000,global_mamba_all,0.01,week,5,02d_mamba_forecasting,True,7,False
7,-1.575092,-2.534032,-1.099988,-0.057809,-0.057532,0.044538,191,True,0.0025,global_mamba_all,0.01,week,5,02d_mamba_forecasting,True,8,False


,cumulative_return,annualized_return,annualized_volatility,periods_per_year,sharpe,sortino,max_drawdown,calmar,turnover,number_of_trades,ticker,model_name,signal_mode,n_rebalances,sample_warning,long_threshold
0,0.036960,0.071744,0.042771,252.0,1.677424,5.143237,-0.029699,2.415689,0.023485,106,__panel__,global_mamba_stationary,overlapping_tranches,863,False,0.01
1,0.057019,0.111671,0.044444,252.0,2.512608,7.126211,-0.032668,3.418352,0.027165,109,__panel__,global_mamba_all,overlapping_tranches,863,False,0.01


,cumulative_return,annualized_return,annualized_volatility,periods_per_year,sharpe,sortino,max_drawdown,calmar,turnover,number_of_trades,ticker,model_name,signal_mode,n_rebalances,sample_warning,long_threshold
0,0.390477,0.933426,0.272225,252.0,3.428872,10.696357,-0.128075,7.288138,0.030159,19,CBOM,global_mamba_stationary,overlapping_tranches,126,False,0.01
1,-0.019390,-0.045067,0.026905,252.0,-1.675061,-1.544979,-0.033633,-1.339978,0.052336,28,MBNK,global_mamba_stationary,overlapping_tranches,107,False,0.01
2,-0.001086,-0.002171,0.002967,252.0,-0.731809,-0.172777,-0.001969,-1.102556,0.006349,4,SBER,global_mamba_stationary,overlapping_tranches,126,False,0.01
3,-0.000849,-0.001698,0.002153,252.0,-0.788728,-0.218467,-0.001447,-1.173521,0.006349,4,SBERP,global_mamba_stationary,overlapping_tranches,126,False,0.01
4,-0.028845,-0.056859,0.049718,252.0,-1.143634,-1.351851,-0.087506,-0.649771,0.044444,28,SVCB,global_mamba_stationary,overlapping_tranches,126,False,0.01
5,0.014413,0.029033,0.017086,252.0,1.699231,1.512650,-0.011062,2.624490,0.015873,10,T,global_mamba_stationary,overlapping_tranches,126,False,0.01
6,-0.074817,-0.144037,0.029134,252.0,-4.943872,-3.279235,-0.074817,-1.925183,0.020635,13,VTBR,global_mamba_stationary,overlapping_tranches,126,False,0.01
7,0.338808,0.792408,0.279820,252.0,2.831844,6.942978,-0.162113,4.887997,0.026984,17,CBOM,global_mamba_all,overlapping_tranches,126,False,0.01
8,-0.017623,-0.041011,0.022520,252.0,-1.821103,-0.961739,-0.019115,-2.145515,0.026168,14,MBNK,global_mamba_all,overlapping_tranches,107,False,0.01
9,0.015619,0.031482,0.014914,252.0,2.110981,3.330761,-0.004086,7.705662,0.034921,22,SBER,global_mamba_all,overlapping_tranches,126,False,0.01


...

,profit_risk_utility,panel_sharpe,mean_ticker_sharpe,cumulative_return,max_drawdown,turnover,number_of_trades,validation_constraints_pass,long_threshold,model_name,selected_threshold,horizon_name,horizon,training_notebook,global_mamba_run
0,-1.944546,-2.697059,-1.762601,-0.016737,-0.028177,0.006570,109,True,0.020,global_mamba_stationary,0.02,month,21,02d_mamba_forecasting,True
1,-2.890604,-3.619696,-2.835948,-0.037835,-0.057433,0.007788,119,True,0.010,global_mamba_stationary,0.02,month,21,02d_mamba_forecasting,True
2,-3.659608,-5.030627,-3.286265,-0.061390,-0.078796,0.006602,105,True,0.005,global_mamba_stationary,0.02,month,21,02d_mamba_forecasting,True
3,-3.717968,-5.341993,-3.157050,-0.078436,-0.094386,0.007544,129,True,0.000,global_mamba_stationary,0.02,month,21,02d_mamba_forecasting,True
4,-1.301407,-1.141359,-1.599404,-0.008295,-0.022297,0.005557,82,True,0.020,global_mamba_all,0.02,month,21,02d_mamba_forecasting,True
5,-2.394318,-2.799030,-2.493505,-0.025331,-0.038754,0.005996,107,True,0.010,global_mamba_all,0.02,month,21,02d_mamba_forecasting,True
6,-2.852201,-3.706841,-2.718295,-0.037905,-0.052053,0.007008,128,True,0.005,global_mamba_all,0.02,month,21,02d_mamba_forecasting,True
7,-3.269545,-4.359815,-3.029056,-0.051402,-0.066949,0.006867,126,True,0.000,global_mamba_all,0.02,month,21,02d_mamba_forecasting,True


,profit_risk_utility,panel_sharpe,mean_ticker_sharpe,cumulative_return,max_drawdown,turnover,number_of_trades,validation_constraints_pass,long_threshold,model_name,selected_threshold,horizon_name,horizon,training_notebook,global_mamba_run,validation_rank,is_validation_selected
4,-1.301407,-1.141359,-1.599404,-0.008295,-0.022297,0.005557,82,True,0.020,global_mamba_all,0.02,month,21,02d_mamba_forecasting,True,1,True
0,-1.944546,-2.697059,-1.762601,-0.016737,-0.028177,0.006570,109,True,0.020,global_mamba_stationary,0.02,month,21,02d_mamba_forecasting,True,2,False
5,-2.394318,-2.799030,-2.493505,-0.025331,-0.038754,0.005996,107,True,0.010,global_mamba_all,0.02,month,21,02d_mamba_forecasting,True,3,False
6,-2.852201,-3.706841,-2.718295,-0.037905,-0.052053,0.007008,128,True,0.005,global_mamba_all,0.02,month,21,02d_mamba_forecasting,True,4,False
1,-2.890604,-3.619696,-2.835948,-0.037835,-0.057433,0.007788,119,True,0.010,global_mamba_stationary,0.02,month,21,02d_mamba_forecasting,True,5,False
7,-3.269545,-4.359815,-3.029056,-0.051402,-0.066949,0.006867,126,True,0.000,global_mamba_all,0.02,month,21,02d_mamba_forecasting,True,6,False
2,-3.659608,-5.030627,-3.286265,-0.061390,-0.078796,0.006602,105,True,0.005,global_mamba_stationary,0.02,month,21,02d_mamba_forecasting,True,7,False
3,-3.717968,-5.341993,-3.157050,-0.078436,-0.094386,0.007544,129,True,0.000,global_mamba_stationary,0.02,month,21,02d_mamba_forecasting,True,8,False


,cumulative_return,annualized_return,annualized_volatility,periods_per_year,sharpe,sortino,max_drawdown,calmar,turnover,number_of_trades,ticker,model_name,signal_mode,n_rebalances,sample_warning,long_threshold
0,0.031539,0.061073,0.009003,252.0,6.783312,18.987348,-0.003366,18.144355,0.003808,59,__panel__,global_mamba_stationary,overlapping_tranches,858,False,0.02
1,0.047171,0.091982,0.016001,252.0,5.748433,12.686301,-0.019615,4.689397,0.004470,77,__panel__,global_mamba_all,overlapping_tranches,858,False,0.02


,cumulative_return,annualized_return,annualized_volatility,periods_per_year,sharpe,sortino,max_drawdown,calmar,turnover,number_of_trades,ticker,model_name,signal_mode,n_rebalances,sample_warning,long_threshold
0,0.187614,0.410426,0.057750,252.0,7.106977,19.427225,-0.013949,29.423178,0.006425,17,CBOM,global_mamba_stationary,overlapping_tranches,126,False,0.02
1,-0.000234,-0.000568,0.000276,252.0,-2.057119,-0.552133,-0.000234,-2.422673,0.000916,2,MBNK,global_mamba_stationary,overlapping_tranches,104,False,0.02
2,0.000935,0.001871,0.001554,252.0,1.204481,11.503937,-0.000194,9.657105,0.002268,6,SBER,global_mamba_stationary,overlapping_tranches,126,False,0.02
3,0.000511,0.001022,0.000831,252.0,1.230955,NaN,-0.000071,14.312963,0.000756,2,SBERP,global_mamba_stationary,overlapping_tranches,126,False,0.02
4,-0.001655,-0.003361,0.002261,252.0,-1.486345,-0.197802,-0.001655,-2.030522,0.000768,2,SVCB,global_mamba_stationary,overlapping_tranches,124,False,0.02
5,0.017224,0.034744,0.009094,252.0,3.820371,4.277019,-0.001660,20.926464,0.005291,14,T,global_mamba_stationary,overlapping_tranches,126,False,0.02
6,0.020374,0.041163,0.020238,252.0,2.033912,1.537455,-0.007923,5.195559,0.006047,16,VTBR,global_mamba_stationary,overlapping_tranches,126,False,0.02
7,0.242941,0.544902,0.092504,252.0,5.890556,10.847200,-0.081362,6.697226,0.004157,11,CBOM,global_mamba_all,overlapping_tranches,126,False,0.02
8,-0.007938,-0.019125,0.014073,252.0,-1.359026,-1.360228,-0.026911,-0.710678,0.005495,12,MBNK,global_mamba_all,overlapping_tranches,104,False,0.02
9,0.022118,0.044726,0.007476,252.0,5.982875,31.571969,-0.000561,79.665518,0.004535,12,SBER,global_mamba_all,overlapping_tranches,126,False,0.02


,check,passed,details
0,global outer splits are available,True,split_rows=7
1,global validation predictions are available,True,rows=1716
2,global test predictions are available,True,rows=1716
3,global cutoffs are chronological,True,"global_validation_start=2025-08-08 00:00:00, g..."
4,ticker split dates are chronological,True,bad_rows=0
5,validation predictions match ticker split dates,True,"out_of_window=0, wrong_role=0"
6,test predictions match ticker split dates,True,"out_of_window=0, wrong_role=0"
7,validation model target dates end before globa...,True,overlap_rows=0
8,final refit target dates end before global tes...,True,overlap_rows=0
9,global final model payloads exist for test pre...,True,missing_models=0


## Comparison Handles

In [7]:
for horizon_name, result in global_mamba_results.items():
    print({
        "horizon": horizon_name,
        "reports_dir": display_path(result["reports_dir"]),
        "test_predictions": display_path(result["reports_dir"] / "test_predictions.parquet"),
        "test_panel_signal_metrics": display_path(result["reports_dir"] / "test_panel_signal_metrics.parquet"),
        "leakage_audit": display_path(result["reports_dir"] / "leakage_audit.parquet"),
    })

{'horizon': 'week', 'reports_dir': 'artifacts/horizons/week/global_mamba/strict_protocol/reports', 'test_predictions': 'artifacts/horizons/week/global_mamba/strict_protocol/reports/test_predictions.parquet', 'test_panel_signal_metrics': 'artifacts/horizons/week/global_mamba/strict_protocol/reports/test_panel_signal_metrics.parquet', 'leakage_audit': 'artifacts/horizons/week/global_mamba/strict_protocol/reports/leakage_audit.parquet'}
{'horizon': 'month', 'reports_dir': 'artifacts/horizons/month/global_mamba/strict_protocol/reports', 'test_predictions': 'artifacts/horizons/month/global_mamba/strict_protocol/reports/test_predictions.parquet', 'test_panel_signal_metrics': 'artifacts/horizons/month/global_mamba/strict_protocol/reports/test_panel_signal_metrics.parquet', 'leakage_audit': 'artifacts/horizons/month/global_mamba/strict_protocol/reports/leakage_audit.parquet'}
